# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# W04 setup — connect to the FlyRank warehouse

import os
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connection: READY")
print("February source: READY")
print("March source: READY")
print("Content dimension: READY")

DuckDB connection: READY
February source: READY
March source: READY
Content dimension: READY


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distribution Review

The key signals used in the content opportunity analysis have different scales and may contain heavy tails.

I inspect February GSC impressions, February GSC clicks, and content age before evaluating their directional relationship with the March observed outcome.

The purpose is descriptive: to understand the distribution of the signals before using them in a ranking or review rule.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build February page-level signals

FEB_SIGNALS = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{FEB}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet('{DIM_CONTENT}')
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days
FROM feb f
JOIN content c
    ON f.client_hash_id = c.client_hash_id
   AND f.content_hash_id = c.content_hash_id
WHERE c.content_created_date IS NOT NULL
""").df()

# Distribution summary
distribution_summary = FEB_SIGNALS[
    ["gsc_impressions", "gsc_clicks", "content_age_days"]
].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
).T

display(distribution_summary)

print("Rows audited:", len(FEB_SIGNALS))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count,mean,std,min,25%,50%,75%,90%,99%,max
gsc_impressions,153559.0,1173.027449,4027.541938,1.0,13.0,119.0,766.0,2770.0,16322.94,203401.0
gsc_clicks,153559.0,3.817458,21.904694,0.0,0.0,0.0,1.0,8.0,61.00,3310.0
content_age_days,153559.0,179.709903,111.480365,0.0,79.0,184.0,235.0,344.0,436.00,463.0


Rows audited: 153559


In [4]:
# Compact distribution checks

for col in ["gsc_impressions", "gsc_clicks", "content_age_days"]:
    print(f"\n{col}")
    print(
        FEB_SIGNALS[col]
        .quantile([0, 0.25, 0.50, 0.75, 0.90, 0.99, 1.00])
    )


gsc_impressions
0.00         1.00
0.25        13.00
0.50       119.00
0.75       766.00
0.90      2770.00
0.99     16322.94
1.00    203401.00
Name: gsc_impressions, dtype: float64

gsc_clicks
0.00       0.0
0.25       0.0
0.50       0.0
0.75       1.0
0.90       8.0
0.99      61.0
1.00    3310.0
Name: gsc_clicks, dtype: float64

content_age_days
0.00      0.0
0.25     79.0
0.50    184.0
0.75    235.0
0.90    344.0
0.99    436.0
1.00    463.0
Name: content_age_days, dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal Test 1 — Content Age

**Signal:** Content age at the February 2026 decision point.

**Test:** Compare the observed March `went_dark` rate across age groups.

**Question:** Does the observed outcome rate differ directionally across younger and older content?

The result is interpreted as an observed association, not evidence that age causes a page to lose clicks.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal test #1 — content age

MAR_OUTCOME = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 1
        ELSE 0
    END AS went_dark
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

signal_df = FEB_SIGNALS.merge(
    MAR_OUTCOME,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

age_test = (
    signal_df.assign(
        age_group=pd.cut(
            signal_df["content_age_days"],
            bins=[-np.inf, 179, 364, np.inf],
            labels=["under_180", "180_364", "365_plus"]
        )
    )
    .groupby("age_group", observed=False)
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean")
    )
    .reset_index()
)

display(age_test)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_group,n,went_dark_rate
0,under_180,66137,0.565054
1,180_364,56902,0.570560
2,365_plus,11199,0.576659


**Verdict: CONFIRMED**

The observed March `went_dark` rate increases modestly across the three content-age groups. The difference is directional but relatively small.

This supports content age as a weak directional signal in this dataset. It does not establish that older content causes lower performance.

### Signal Test 2 — Search Impressions

**Signal:** February 2026 GSC impressions.

**Test:** Compare the observed March `went_dark` rate across impression bands.

**Question:** Does lower observed search visibility correspond to a different March outcome rate?

The result is interpreted as a directional association rather than a causal effect.

In [6]:
# Signal test #2 — February impressions

impression_test = (
    signal_df.assign(
        impression_group=pd.cut(
            signal_df["gsc_impressions"],
            bins=[-np.inf, 99, 999, np.inf],
            labels=["under_100", "100_999", "1000_plus"]
        )
    )
    .groupby("impression_group", observed=False)
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean")
    )
    .reset_index()
)

display(impression_test)

,impression_group,n,went_dark_rate
0,under_100,57536,0.872619
1,100_999,43794,0.533110
2,1000_plus,32908,0.083293


**Verdict: CONFIRMED**

The observed March `went_dark` rate changes strongly across the impression groups. Pages with fewer February impressions have a substantially higher observed March zero-click rate in this dataset.

This is a strong directional signal for review prioritization, but it does not establish that increasing impressions or changing a page would cause the March outcome to improve.

### Signal Test 3 — Search Clicks

**Signal:** February 2026 GSC clicks.

**Test:** Compare the observed March `went_dark` rate across February click bands.

**Question:** Does lower observed search traffic correspond to a different March outcome rate?

This test is descriptive and does not treat the relationship as causal.

In [7]:
# Signal test #3 — February clicks

click_test = (
    signal_df.assign(
        click_group=pd.cut(
            signal_df["gsc_clicks"],
            bins=[-np.inf, 0, 9, 99, np.inf],
            labels=["0", "1_9", "10_99", "100_plus"]
        )
    )
    .groupby("click_group", observed=False)
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean")
    )
    .reset_index()
)

display(click_test)

,click_group,n,went_dark_rate
0,0,80872,0.809229
1,1_9,40033,0.269678
2,10_99,12598,0.004366
3,100_plus,735,0.000000


**Verdict: CONFIRMED**

The observed March outcome rate changes directionally across February click groups. Lower February click activity is associated with a higher observed March `went_dark` rate in this evaluation dataset.

This supports clicks as a directional ranking signal, while the analysis remains observational.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-Linked Test — Low-Impression Review Flag

The Week-4 baseline uses low February impressions as a review signal.

The specific rule is:

`gsc_impressions < 100`

The methodology question is whether pages meeting this rule actually show a different observed March outcome rate from pages above the threshold.

The test is descriptive and checks whether the rule's directional assumption is supported by the observed data.

In [8]:
# Flag-linked test — February impressions below 100

flag_test = (
    signal_df.assign(
        low_impression_flag=signal_df["gsc_impressions"] < 100
    )
    .groupby("low_impression_flag")
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean")
    )
    .reset_index()
)

display(flag_test)

low_rate = flag_test.loc[
    flag_test["low_impression_flag"] == True,
    "went_dark_rate"
].iloc[0]

other_rate = flag_test.loc[
    flag_test["low_impression_flag"] == False,
    "went_dark_rate"
].iloc[0]

print("Low-impression observed rate:", round(low_rate, 4))
print("Other pages observed rate:", round(other_rate, 4))
print("Difference:", round(low_rate - other_rate, 4))

,low_impression_flag,n,went_dark_rate
0,False,76702,0.340122
1,True,57536,0.872619


Low-impression observed rate: 0.8726
Other pages observed rate: 0.3401
Difference: 0.5325


### Flag Verdict

**Verdict: CONFIRMED**

The observed March `went_dark` rate is higher among pages with fewer than 100 February impressions than among pages above the threshold.

This supports the low-impression rule as a directional review signal in this dataset. It should still be treated as a prioritization flag rather than an automatic content action.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### Practical Interpretation

The audit shows that February search visibility signals, especially impressions, have meaningful directional relationships with the observed March outcome in this dataset. Content age shows a smaller directional difference, while the click signal is evaluated separately rather than assumed to be useful. A content team can therefore use these signals to prioritize human review, while treating the flags as decision-support indicators rather than automatic instructions or causal predictions.

In [9]:
# Final signal-audit summary

signal_summary = pd.DataFrame({
    "signal": [
        "content_age_days",
        "gsc_impressions",
        "gsc_clicks",
        "low_impression_flag"
    ],
    "test": [
        "Observed March went_dark rate by age group",
        "Observed March went_dark rate by impression group",
        "Observed March went_dark rate by click group",
        "Observed March went_dark rate below vs above 100 impressions"
    ]
})

display(signal_summary)

,signal,test
0,content_age_days,Observed March went_dark rate by age group
1,gsc_impressions,Observed March went_dark rate by impression group
2,gsc_clicks,Observed March went_dark rate by click group
3,low_impression_flag,Observed March went_dark rate below vs above 1...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.